# Notebook 1: Prepare BBBC036v1 standardized dataset

Per `chemical_surrogate_study/PLAN.md`. Reuses the validated TVN-corrected wells and activity
labels from the earlier `08_activity_confound_study.ipynb` run; recomputes toxicity **restricted
to active compounds + negative controls only** (methodological consistency with notebook 2 -
toxicity among inactives is vanishingly rare, 3/10,680 = 0.03% in the earlier full-population run,
so restricting the test loses essentially no information).

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from scipy import linalg
from sklearn.preprocessing import StandardScaler
from copairs import map as copairs_map
from copairs.matching import assign_reference_index

# Assumes the notebook runs with its own directory (chemical_surrogate_study/notebooks/) as the
# working directory, which is Jupyter's default when opening a notebook.
PROJECT_ROOT = Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()
STUDY_DIR = PROJECT_ROOT / "chemical_surrogate_study"
DATA_DIR = STUDY_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OLD_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "full_pipeline"  # optional local speedup only, see README
RAW_PROFILE_CSV = PROJECT_ROOT / "data" / "train_set_30kcpds_normalized_profiles.csv.gz"
CELLCLIP_SPLIT_REPO_ID = "suinleelab/CellCLIP"

RANDOM_SEED = 42
SAMPLES_NEGCON = 190
NULL_SIZE = 10000
P_THRESHOLD = 0.05
CELLCOUNT_COL = "Cells_Number_Object_Number"

print(f"Study dir: {STUDY_DIR}")

## Shared helper functions (reused verbatim from the validated prior pipeline)

In [2]:
def centerscale_on_controls(embeddings, metadata, pert_col, control_key, batch_col=None):
    embeddings = embeddings.copy()
    if batch_col is not None:
        for batch in metadata[batch_col].unique():
            batch_ind = (metadata[batch_col] == batch).to_numpy()
            batch_control_ind = batch_ind & (metadata[pert_col] == control_key).to_numpy()
            embeddings[batch_ind] = StandardScaler().fit(embeddings[batch_control_ind]).transform(embeddings[batch_ind])
        return embeddings
    control_ind = (metadata[pert_col] == control_key).to_numpy()
    return StandardScaler().fit(embeddings[control_ind]).transform(embeddings)


def tvn_on_controls(embeddings, metadata, pert_col, control_key, batch_col=None):
    from sklearn.decomposition import PCA
    embeddings = centerscale_on_controls(embeddings, metadata, pert_col, control_key)
    ctrl_ind = (metadata[pert_col] == control_key).to_numpy()
    embeddings = PCA().fit(embeddings[ctrl_ind]).transform(embeddings)
    embeddings = centerscale_on_controls(embeddings, metadata, pert_col, control_key, batch_col)
    target_cov = np.cov(embeddings[ctrl_ind], rowvar=False, ddof=1) + 0.5 * np.eye(embeddings.shape[1])
    if batch_col is not None:
        target_cov_pow = linalg.fractional_matrix_power(target_cov, 0.5)
        for batch in metadata[batch_col].unique():
            batch_ind = (metadata[batch_col] == batch).to_numpy()
            batch_control_ind = batch_ind & ctrl_ind
            source_cov = np.cov(embeddings[batch_control_ind], rowvar=False, ddof=1) + 0.5 * np.eye(embeddings.shape[1])
            embeddings[batch_ind] = np.matmul(embeddings[batch_ind], linalg.fractional_matrix_power(source_cov, -0.5))
            embeddings[batch_ind] = np.matmul(embeddings[batch_ind], target_cov_pow)
    return embeddings


def sample_controls_per_batch(controls_df, batch_col, samples_negcon, random_state):
    return (controls_df.sample(frac=1, random_state=random_state)
            .groupby(batch_col, group_keys=False).head(samples_negcon))


def copairs_phenotypic_activity(features_and_metadata, feature_cols, control_mask, control_query,
                                 control_perturbation, samples_negcon, null_size, random_state,
                                 perturbation_col, batch_col, threshold=P_THRESHOLD, cache_dir=None,
                                 distance="cosine"):
    """distance='euclidean' for a single scalar feature (cell count) - cosine on a 1-D feature
    collapses to sign(x), discarding all magnitude information (caught earlier: gave a bogus
    99.5% 'toxic' rate)."""
    control_mask = pd.Series(control_mask, index=features_and_metadata.index).fillna(False).astype(bool)
    controls_df = features_and_metadata.loc[control_mask].copy()
    profiles_df = features_and_metadata.loc[~control_mask].copy()
    sampled_controls = sample_controls_per_batch(controls_df, batch_col, samples_negcon, random_state)
    subset = pd.concat([profiles_df, sampled_controls], axis=0).reset_index(drop=True)

    reference_col = "reference_index"
    df_activity = assign_reference_index(subset, control_query, reference_col=reference_col, default_value=-1)
    feature_cols = list(feature_cols)
    features = df_activity[feature_cols].to_numpy(dtype=np.float32, copy=False)
    metadata_cols = df_activity.columns.difference(feature_cols)

    sameby = [perturbation_col, reference_col]
    activity_ap = copairs_map.average_precision(
        df_activity[metadata_cols], features, pos_sameby=sameby, pos_diffby=[],
        neg_sameby=[batch_col], neg_diffby=sameby, distance=distance,
    )
    activity_ap = activity_ap.loc[:, ~activity_ap.columns.duplicated()].copy()
    activity_ap = activity_ap.query(f"{perturbation_col} != @control_perturbation").copy()

    activity_map = copairs_map.mean_average_precision(
        activity_ap, sameby, null_size=null_size, threshold=threshold, seed=random_state,
        cache_dir=Path(cache_dir) if cache_dir is not None else None,
    )
    activity_map = activity_map.query(f"{perturbation_col} != @control_perturbation").copy()
    return activity_map


def load_split(name):
    path = hf_hub_download(CELLCLIP_SPLIT_REPO_ID, f"datasplit1-{name}.csv")
    df = pd.read_csv(path, usecols=["BROAD_ID", "SMILES", "INCHIKEY"])
    df["split"] = name
    return df

## 1. Official split + population

In [3]:
official = pd.concat([load_split(s) for s in ["train", "val", "test"]], axis=0, ignore_index=True)
official = official.drop_duplicates("BROAD_ID").reset_index(drop=True)
print(f"Official population: {len(official):,} compounds")

Official population: 10,683 compounds


## 2. TVN-corrected wells (cell count excluded from the feature set before whitening)

Reuses the validated cache if present (the full recompute takes ~21 min: 1261.6s for TVN alone).

In [4]:
TVN_NOCC_CACHE = OLD_OUTPUT_DIR / "tvn_corrected_wells_nocellcount.parquet"

if TVN_NOCC_CACHE.exists():
    print("Loading cached cell-count-excluded TVN wells...")
    tvn_wells_nocc = pd.read_parquet(TVN_NOCC_CACHE)
    feature_cols_no_cc = [c for c in tvn_wells_nocc.columns
                           if c not in ("BROAD_ID", "plate", "Metadata_Well", "is_control", "perturbation")]
else:
    print("No cache found - recomputing from raw profile CSV (this will take ~20-25 min)...")
    official_ids = set(official["BROAD_ID"])
    meta_cols = ["Metadata_broad_sample", "Metadata_Plate", "Metadata_Well", "BROAD_ID", "CPD_NAME", "SMILES_standard", "structure"]
    well_cols = pd.read_csv(RAW_PROFILE_CSV, nrows=0).columns.tolist()
    feature_cols_raw = [c for c in well_cols if c not in meta_cols and not c.startswith("Unnamed")]
    raw = pd.read_csv(RAW_PROFILE_CSV, usecols=meta_cols + feature_cols_raw)
    raw["plate"] = raw["Metadata_Plate"].astype(str)
    raw["is_control"] = raw["Metadata_broad_sample"].astype(str).eq("DMSO")
    raw = raw[raw["is_control"] | raw["BROAD_ID"].isin(official_ids)].reset_index(drop=True)
    raw["perturbation"] = np.where(raw["is_control"], "DMSO", raw["BROAD_ID"])
    feature_cols_no_cc = [c for c in feature_cols_raw if c != CELLCOUNT_COL]

    t0 = time.time()
    tvn_features = tvn_on_controls(raw[feature_cols_no_cc].to_numpy(dtype=np.float64), raw, "is_control", True, batch_col="plate")
    tvn_wells_nocc = pd.concat([
        raw[["BROAD_ID", "plate", "Metadata_Well", "is_control", "perturbation"]].reset_index(drop=True),
        pd.DataFrame(tvn_features.astype(np.float32), columns=feature_cols_no_cc),
    ], axis=1)
    tvn_wells_nocc.to_parquet(TVN_NOCC_CACHE)
    print(f"TVN done in {time.time()-t0:.1f}s")

print(f"TVN wells: {len(tvn_wells_nocc):,}, compounds: {tvn_wells_nocc.loc[~tvn_wells_nocc['is_control'],'BROAD_ID'].nunique():,}")

Loading cached cell-count-excluded TVN wells...


TVN wells: 73,946, compounds: 10,683


## 3. Raw per-plate cell count (needed for toxicity, kept separate from the whitened profile)

In [5]:
CELLCOUNT_WELLS_CACHE = STUDY_DIR / "data" / "_bbbc_raw_cellcount_wells.parquet"

if CELLCOUNT_WELLS_CACHE.exists():
    cc_raw_wells = pd.read_parquet(CELLCOUNT_WELLS_CACHE)
else:
    official_ids = set(official["BROAD_ID"])
    meta_cols = ["Metadata_broad_sample", "Metadata_Plate", "Metadata_Well", "BROAD_ID"]
    raw_cc = pd.read_csv(RAW_PROFILE_CSV, usecols=meta_cols + [CELLCOUNT_COL])
    raw_cc["plate"] = raw_cc["Metadata_Plate"].astype(str)
    raw_cc["is_control"] = raw_cc["Metadata_broad_sample"].astype(str).eq("DMSO")
    raw_cc = raw_cc[raw_cc["is_control"] | raw_cc["BROAD_ID"].isin(official_ids)].reset_index(drop=True)
    raw_cc["perturbation"] = np.where(raw_cc["is_control"], "DMSO", raw_cc["BROAD_ID"])
    cc_raw_wells = raw_cc[["BROAD_ID", "plate", "Metadata_Well", "is_control", "perturbation", CELLCOUNT_COL]]
    cc_raw_wells.to_parquet(CELLCOUNT_WELLS_CACHE)

print(f"Cell-count wells: {len(cc_raw_wells):,}")

Cell-count wells: 73,946


## 4. Activity label

Reuses the validated result if the population matches (identical TVN wells + methodology - cosine
distance, full cell-count-excluded profile).

In [6]:
OLD_LABELS_CACHE = OLD_OUTPUT_DIR / "bbbc036v1_confound_labels.csv"

if OLD_LABELS_CACHE.exists():
    old_labels = pd.read_csv(OLD_LABELS_CACHE)
    if set(old_labels["BROAD_ID"]) >= set(tvn_wells_nocc.loc[~tvn_wells_nocc["is_control"], "BROAD_ID"].unique()):
        print("Reusing cached activity labels (population matches)...")
        active_labels = old_labels[["BROAD_ID", "activity_map", "activity_p", "is_active"]].copy()
    else:
        old_labels = None
else:
    old_labels = None

if old_labels is None:
    print("Recomputing activity labels...")
    t0 = time.time()
    active_map = copairs_phenotypic_activity(
        tvn_wells_nocc, feature_cols=feature_cols_no_cc, control_mask=tvn_wells_nocc["is_control"],
        control_query="is_control == True", control_perturbation="DMSO",
        samples_negcon=SAMPLES_NEGCON, null_size=NULL_SIZE, random_state=RANDOM_SEED,
        perturbation_col="perturbation", batch_col="plate",
        cache_dir=STUDY_DIR / "data" / "_bbbc_activity_null_cache",
    )
    print(f"Active-label copairs test done in {time.time()-t0:.1f}s")
    active_labels = active_map.drop_duplicates("perturbation")[
        ["perturbation", "mean_average_precision", "corrected_p_value", "below_corrected_p"]
    ].rename(columns={"perturbation": "BROAD_ID", "mean_average_precision": "activity_map",
                       "corrected_p_value": "activity_p", "below_corrected_p": "is_active"})

print(f"Active: {active_labels['is_active'].sum():,} / {len(active_labels):,} ({active_labels['is_active'].mean():.1%})")

Recomputing activity labels...


  0%|          | 0/5 [00:00<?, ?it/s]

 20%|██        | 1/5 [00:00<00:02,  1.35it/s]

 40%|████      | 2/5 [00:00<00:01,  2.33it/s]

  0%|          | 0/153 [00:00<?, ?it/s]

...

 37%|███▋      | 3983/10680 [00:00<00:00, 19441.69it/s]

 56%|█████▌    | 5929/10680 [00:00<00:00, 19439.17it/s]

 74%|███████▎  | 7874/10680 [00:00<00:00, 18707.58it/s]

 92%|█████████▏| 9826/10680 [00:00<00:00, 18987.48it/s]

Active-label copairs test done in 13.1s
Active: 9,746 / 10,680 (91.3%)


## 5. Toxicity label - restricted to active compounds + negative controls only

New methodology per PLAN.md: toxicity is only tested among active compounds (inactive-and-toxic
was 3/10,680 = 0.03% in the earlier full-population run, so this loses essentially no information
while making the JUMP-CP notebook's toxicity fetch tractable at scale). Inactive compounds get
`is_toxic = False` (untested) / `toxicity_map, toxicity_p = NaN`.

In [7]:
active_ids = set(active_labels.loc[active_labels["is_active"], "BROAD_ID"])
print(f"Restricting toxicity test to {len(active_ids):,} active compounds + negative controls")

cc_subset = cc_raw_wells[cc_raw_wells["is_control"] | cc_raw_wells["BROAD_ID"].isin(active_ids)].reset_index(drop=True)
cellcount_std = centerscale_on_controls(cc_subset[[CELLCOUNT_COL]].to_numpy(dtype=np.float64), cc_subset,
                                         "is_control", True, batch_col="plate")
cc_wells_std = pd.concat([
    cc_subset[["BROAD_ID", "plate", "Metadata_Well", "is_control", "perturbation"]].reset_index(drop=True),
    pd.DataFrame(cellcount_std.astype(np.float32), columns=[CELLCOUNT_COL]),
], axis=1)

t0 = time.time()
toxic_map = copairs_phenotypic_activity(
    cc_wells_std, feature_cols=[CELLCOUNT_COL], control_mask=cc_wells_std["is_control"],
    control_query="is_control == True", control_perturbation="DMSO",
    samples_negcon=SAMPLES_NEGCON, null_size=NULL_SIZE, random_state=RANDOM_SEED,
    perturbation_col="perturbation", batch_col="plate",
    cache_dir=STUDY_DIR / "data" / "_bbbc_toxicity_null_cache",
    distance="euclidean",
)
print(f"Toxic-label copairs test done in {time.time()-t0:.1f}s")
toxic_labels = toxic_map.drop_duplicates("perturbation")[
    ["perturbation", "mean_average_precision", "corrected_p_value", "below_corrected_p"]
].rename(columns={"perturbation": "BROAD_ID", "mean_average_precision": "toxicity_map",
                   "corrected_p_value": "toxicity_p", "below_corrected_p": "is_toxic"})
print(f"Toxic (among actives): {toxic_labels['is_toxic'].sum():,} / {len(toxic_labels):,} ({toxic_labels['is_toxic'].mean():.1%})")

labels = active_labels.merge(toxic_labels, on="BROAD_ID", how="left")
labels["is_toxic"] = labels["is_toxic"].fillna(False)
labels["is_active_and_toxic"] = labels["is_active"] & labels["is_toxic"]
labels["is_active_not_toxic"] = labels["is_active"] & ~labels["is_toxic"]

print(labels[["is_active", "is_active_and_toxic", "is_active_not_toxic"]].sum())
print("\n2x2 contingency (active x toxic):")
print(pd.crosstab(labels["is_active"], labels["is_toxic"]))

Restricting toxicity test to 9,746 active compounds + negative controls


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/140 [00:00<?, ?it/s]

  0%|          | 0/29 [00:00<?, ?it/s]

  0%|          | 0/9746 [00:00<?, ?it/s]

...

 55%|█████▍    | 5327/9746 [00:00<00:00, 17045.45it/s]

 72%|███████▏  | 7035/9746 [00:00<00:00, 16810.26it/s]

 90%|████████▉ | 8723/9746 [00:00<00:00, 16833.57it/s]

Toxic-label copairs test done in 2.0s
Toxic (among actives): 615 / 9,746 (6.3%)
is_active              9746
is_active_and_toxic     615
is_active_not_toxic    9131
dtype: int64

2x2 contingency (active x toxic):
is_toxic   False  True 
is_active              
False        934      0
True        9131    615


/var/folders/ch/2fy5zt9x53vfq0jsm_n8ml0w0000gn/T/ipykernel_63107/1710176362.py:29: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels["is_toxic"] = labels["is_toxic"].fillna(False)


## 6. `batch_id`: connected components of co-occurring plates

In [8]:
compound_wells = tvn_wells_nocc[~tvn_wells_nocc["is_control"]]
plates_list = compound_wells["plate"].unique().tolist()
plate_idx = {p: i for i, p in enumerate(plates_list)}
parent = list(range(len(plates_list)))


def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x


def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb


for _, sub in compound_wells.groupby("BROAD_ID")["plate"]:
    idxs = [plate_idx[p] for p in sub]
    for i in idxs[1:]:
        union(idxs[0], i)

comp_of_plate = {p: find(plate_idx[p]) for p in plates_list}
plate_batch_of_compound = compound_wells.assign(
    plate_batch=compound_wells["plate"].map(comp_of_plate)
).groupby("BROAD_ID")["plate_batch"].first()
print(f"Plate-batch split: {plate_batch_of_compound.nunique()} connected-component batches over {compound_wells['plate'].nunique()} plates")

Plate-batch split: 95 connected-component batches over 398 plates


## 7. Aggregate morphology to compound level, assemble standardized schema, write

In [9]:
morphology_compound = tvn_wells_nocc[~tvn_wells_nocc["is_control"]].groupby("BROAD_ID")[feature_cols_no_cc].median()

population = official[official["BROAD_ID"].isin(morphology_compound.index) &
                       official["BROAD_ID"].isin(labels["BROAD_ID"])].reset_index(drop=True)
population = population.merge(labels, on="BROAD_ID", how="left")
population["batch_id"] = population["BROAD_ID"].map(plate_batch_of_compound)
population = population.rename(columns={"BROAD_ID": "compound_id", "split": "cloome_split"})

morph_matrix = morphology_compound.reindex(population["compound_id"]).to_numpy(dtype=np.float32)
morph_matrix = np.nan_to_num(morph_matrix, nan=0.0, posinf=0.0, neginf=0.0)
morph_cols = [f"morph_{i}" for i in range(morph_matrix.shape[1])]
morph_df = pd.DataFrame(morph_matrix, columns=morph_cols, index=population.index)

standardized = pd.concat([
    population[["compound_id", "SMILES", "is_active", "is_active_and_toxic", "is_active_not_toxic",
                "activity_map", "activity_p", "toxicity_map", "toxicity_p", "batch_id", "cloome_split"]],
    morph_df,
], axis=1)

print(f"Final standardized population: {len(standardized):,} compounds, {len(morph_cols)} morphology features")
print(standardized[["is_active", "is_active_and_toxic", "is_active_not_toxic"]].mean())

OUT_PATH = DATA_DIR / "bbbc036v1_standardized.parquet"
standardized.to_parquet(OUT_PATH)
print(f"\nSaved -> {OUT_PATH}")

Final standardized population: 10,680 compounds, 1448 morphology features
is_active              0.912547
is_active_and_toxic    0.057584
is_active_not_toxic    0.854963
dtype: float64



Saved -> /Users/telio/chemical-surrogate-limits/chemical_surrogate_study/data/bbbc036v1_standardized.parquet
